#silver 
    - silver.sessions - merge into
    - silver.session_faults
    - silver.quarantine_sessions
    - silver.stations
    - silver.tariffs
    - silver.chargers - 1
    - silver.members - 2

In [0]:
dbutils.widgets.text("catalog","sj20030614")
catalog = dbutils.widgets.get("catalog")

#session

#stations

In [0]:
station_df = spark.table(f"{catalog}.bronze.stations")

In [0]:
from pyspark.sql import functions as F

silver_stations = (
    station_df
    .withColumn("station_id", F.upper(F.trim("station_id")))
    .withColumn("station_name", F.trim("station_name"))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("site_type", F.trim("site_type"))
    .withColumn("opened_on", F.to_date("opened_on", "yyyy-MM-dd"))
)


(silver_stations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.stations"))


spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.stations
IS 'Cleaned stations data'
""")

#traffis

In [0]:
tariffs_df = spark.table(f"{catalog}.bronze.tariffs")

In [0]:
from pyspark.sql import functions as F

silver_tariffs = (
    tariffs_df
    .withColumn("connector_type", F.upper(F.trim("connector_type")))
    .withColumn("time_band", F.upper(F.trim("time_band")))
    .withColumn("rate_per_kwh", F.col("rate_per_kwh").cast("decimal(10,2)"))
    .withColumn("idle_fee_per_min", F.col("idle_fee_per_min").cast("decimal(10,2)"))
    .withColumn("free_idle_minutes", F.col("free_idle_minutes").cast("int"))
)


(silver_tariffs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tariffs"))


spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.tariffs
IS 'Cleaned stations data'
""")

#chargers

In [0]:
chargers_df = spark.table(f"{catalog}.bronze.chargers")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.chargers (
  charger_id                 STRING,
  station_id                 STRING,
  connector_type             STRING,
  max_kw                     DECIMAL(6,1),
  is_active                  BOOLEAN,
  commissioned_on            DATE,
  last_changed_snapshot_date DATE
)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

silver_chargers = (
    chargers_df
    .withColumn("charger_id",       F.upper(F.trim("charger_id")))
    .withColumn("station_id",        F.upper(F.trim("station_id")))
    .withColumn("connector_type",    F.upper(F.trim(F.col("connector.type"))))
    .withColumn("max_kw",            F.col("connector.max_kw").cast("decimal(6,1)"))
    .withColumn("is_active",         F.col("is_active").cast("boolean"))
    .withColumn("commissioned_on",   F.to_date("commissioned_on", "yyyy-MM-dd"))
    .withColumn("snapshot_date",     F.to_date("snapshot_date"))
)


w = Window.partitionBy("charger_id").orderBy(F.col("snapshot_date").desc_nulls_last())

latest_chargers = (
    silver_chargers
    .filter(F.col("charger_id").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("snapshot_date", "last_changed_snapshot_date")
    .select(
        "charger_id", "station_id", "connector_type",
        "max_kw", "is_active", "commissioned_on",
        "last_changed_snapshot_date"
    )
)


latest_chargers.createOrReplaceTempView("latest_chargers")


In [0]:

spark.sql(f"""
MERGE INTO {catalog}.silver.chargers t
USING latest_chargers s
ON t.charger_id = s.charger_id
WHEN MATCHED AND (
    t.station_id IS DISTINCT FROM s.station_id OR
    t.connector_type   IS DISTINCT FROM s.connector_type   OR
    t.max_kw IS DISTINCT FROM s.max_kw OR
    t.is_active IS DISTINCT FROM s.is_active OR
    t.commissioned_on IS DISTINCT FROM s.commissioned_on
)
THEN UPDATE SET 
 t.station_id = s.station_id,
 t.connector_type = s.connector_type,
 t.max_kw = s.max_kw,
 t.is_active = s.is_active,
 t.commissioned_on = s.commissioned_on,
 t.last_changed_snapshot_date = s.last_changed_snapshot_date
 
WHEN NOT MATCHED
  THEN INSERT *
""")


In [0]:

print(f"silver.chargers: {spark.table(f'{catalog}.silver.chargers').count()} rows")

#members

In [0]:
members_df = spark.table(f"""{catalog}.bronze.members""")

In [0]:
from pyspark.sql import functions as F

# 1. Standardize (city as stations: initcap+trim)
standardized = (
    members_df
    .withColumn("member_id", F.upper(F.trim("member_id")))
    .withColumn("full_name", F.trim("full_name"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("phone", F.regexp_replace(F.col("phone"), "[^0-9]", ""))
    .withColumn("phone", F.right(F.col("phone"), F.lit(10)))
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("plan", F.upper(F.trim("plan")))
    .withColumn("joined_on", F.coalesce(
        F.try_to_date(F.col("joined_on"), F.lit("yyyy-MM-dd")),
        F.try_to_date(F.col("joined_on"), F.lit("dd/MM/yyyy"))
    ))
    .withColumn("snapshot_date", F.try_to_date(
        F.regexp_extract("_source_file", r"members_(\d{4}-\d{2}-\d{2})\.csv", 1),
        F.lit("yyyy-MM-dd")
    ))
)

# 2. Deduplicate identical rows (rule 1)
deduped = standardized.dropDuplicates([
    "member_id", "full_name", "email", "phone", "city", "plan", "joined_on", "snapshot_date"
])

# 2b. Keep only the latest snapshot per member (rule 2: ordered by snapshot_date)
from pyspark.sql import Window

_w = Window.partitionBy("member_id").orderBy(F.col("snapshot_date").desc_nulls_last())
latest_per_member = (
    deduped
    .withColumn("rn", F.row_number().over(_w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 3. Build SCD Type 2 versions (rule 2: ordered by snapshot_date per member)
scd2 = (
    latest_per_member
    .select(
        "member_id", "full_name", "email", "phone", "city", "plan", "joined_on",
        F.col("snapshot_date").alias("effective_from"),
        F.lit(None).cast("date").alias("effective_to"),
        F.lit(True).alias("is_current"),
    )
)

# 4. Ensure table has SCD Type 2 schema
_actual_cols = set(spark.table(f"{catalog}.silver.members").columns)
if "effective_from" not in _actual_cols:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.silver.members (
      member_id      STRING,
      full_name      STRING,
      email          STRING,
      phone          STRING,
      city           STRING,
      plan           STRING,
      joined_on      DATE,
      effective_from DATE,
      effective_to   DATE,
      is_current     BOOLEAN
    )
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """)

scd2.createOrReplaceTempView("scd2_members")

# 5a. Close current version on plan/city change (rule 3)
spark.sql(f"""
MERGE INTO {catalog}.silver.members t
USING scd2_members s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND (
    t.city IS DISTINCT FROM s.city OR
    t.plan IS DISTINCT FROM s.plan
)
THEN UPDATE SET
    t.effective_to = s.effective_from,
    t.is_current = false
""")

# 5b. Overwrite on name/email/phone change (rule 4),
#     insert new version after close + new members (rule 5)
spark.sql(f"""
MERGE INTO {catalog}.silver.members t
USING scd2_members s
ON t.member_id = s.member_id AND t.is_current = true
WHEN MATCHED AND (
    t.full_name IS DISTINCT FROM s.full_name OR
    t.email IS DISTINCT FROM s.email OR
    t.phone IS DISTINCT FROM s.phone
) AND (
    t.city = s.city AND t.plan = s.plan
)
THEN UPDATE SET
    t.full_name = s.full_name,
    t.email = s.email,
    t.phone = s.phone
WHEN NOT MATCHED
  THEN INSERT *
""")

print(f"silver.members: {spark.table(f'{catalog}.silver.members').count()} rows")

#session

In [0]:
df = spark.table(f"{catalog}.bronze.sessions")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window


#standarization
df = (
    df.withColumn("session_id", upper(trim(col("session_id"))).cast("string"))
    .withColumn("charger_id", upper(trim(col("charger_id"))).cast("string"))
    .withColumn("member_id", upper(trim(col("member_id"))).cast("string"))
    .withColumn("started_at", coalesce(
        try_to_timestamp(col("started_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("started_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))
    .withColumn("ended_at", coalesce(
        try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))
    .withColumn("session_date", to_date(col("started_at")))
    .withColumn("energy_kwh", 
                when(
                    (col("meter.start_kwh").isNotNull()) &
                    (col("meter.end_kwh").isNotNull()),
                    col("meter.end_kwh") - col("meter.start_kwh")
                )
                .otherwise(None)
                .cast("double")
    )
    .withColumn("status",upper(trim(col("status"))))
    .withColumn("idle_minutes", coalesce(col("idle_minutes"), lit(0)).cast("int"))
    .withColumn("faults", transform(col("faults"), lambda x: struct(
        x["at"].alias("at"),
        upper(trim(x["code"])).alias("code")
    )))
    .withColumn("member_id", coalesce(col("member_id"), lit("GUESTS")))
    .withColumn("duration_minutes", 
                when(
                    (col("ended_at").isNotNull()) &
                    (col("started_at").isNotNull()),
                    unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))
                )
                .cast("integer")
    )
    .withColumn("time_band", when(
        (hour(col("started_at")) >= 18) & (hour(col("started_at")) <= 21), lit("PEAK")
    ).otherwise(lit("OFF_PEAK")))      
)

#latest
w = Window.partitionBy("session_id").orderBy(
    col("updated_at").desc_nulls_last()
)

latest = (
    df
    .filter(col("session_id").isNotNull())
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

#valid_ids
valid_charger_id = [
    row[0]
    for row in spark.table(f"{catalog}.silver.chargers")
    .select("charger_id")
    .distinct()
    .collect()
]

valid_member_id = [
    row[0]
    for row in spark.table(f"{catalog}.silver.members")
    .select("member_id")
    .distinct()
    .collect()
]

#validation 
checked = latest.withColumn(
    "validation_error",
    when(
        col("charger_id").isNull() |
        ~col("charger_id").isin(valid_charger_id),
        "UNKNOWN_CHARGER"
    )
    .when(
        ~col("member_id").isin(valid_member_id),
        "UNKNOWN_MEMBER"
    )
    .when(
        col("started_at").isNull(),
        "INVALID_TIMESTAMP"
    )
    .when(
        (col("status").isin("COMPLETED", "FAILED")) &
        (col("ended_at").isNull() | (col("ended_at") <= col("started_at"))),
        "INVALID_DURATION"
    )
    .when(
        (col("status") == "COMPLETED") &
        (col("energy_kwh").isNull() | (col("energy_kwh") < 0)),
        "INVALID_METER"
    )
)

valid_sessions = checked.filter(col("validation_error").isNull())

quarantine_sessions = checked.filter(col("validation_error").isNotNull())


In [0]:
valid_sessions.createOrReplaceTempView("valid_sessions")

In [0]:
# silver.sessions (valid sessions): session_id, charger_id, member_id STRING;
# started_at, ended_at TIMESTAMP; session_date DATE; status, time_band STRING;
# energy_kwh DECIMAL(10,3); duration_minutes, idle_minutes INT;
# amount_paid DECIMAL(10,2) (from payment.amount_inr); updated_at TIMESTAMP.

(valid_sessions
    .select(
        "session_id", "charger_id", "member_id",
        "started_at", "ended_at", "session_date",
        "status", "time_band",
        col("energy_kwh").cast("decimal(10,3)"),
        col("duration_minutes").cast("int"),
        col("idle_minutes").cast("int"),
        col("payment.amount_inr").cast("decimal(10,2)").alias("amount_paid"),
        "updated_at",
    )
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.sessions")
)

# silver.session_faults (one row per fault of a valid session):
# session_id STRING, fault_seq INT (1-based), fault_code STRING

(valid_sessions
    .select("session_id", posexplode(col("faults")))
    .select(
        "session_id",
        (col("pos") + 1).alias("fault_seq"),
        col("col.code").alias("fault_code"),
    )
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.session_faults")
)

# silver.quarantine_sessions (one row per rejected session):
# session_id, reject_reason STRING; updated_at TIMESTAMP;
# raw_record STRING (full row as JSON); quarantined_at TIMESTAMP

(quarantine_sessions
    .select(
        "session_id",
        col("validation_error").alias("reject_reason"),
        "updated_at",
        to_json(struct("*")).alias("raw_record"),
        current_timestamp().alias("quarantined_at"),
    )
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.quarantine_sessions")
)

print(f"silver.sessions:       {spark.table(f'{catalog}.silver.sessions').count()} rows")
print(f"silver.session_faults: {spark.table(f'{catalog}.silver.session_faults').count()} rows")
print(f"silver.quarantine:      {spark.table(f'{catalog}.silver.quarantine_sessions').count()} rows")

In [0]:
spark.sql(f"""

MERGE INTO {catalog}.silver.sessions t
USING valid_sessions s
ON t.session_id = s.session_id
WHEN MATCHED THEN
  UPDATE SET
    t.charger_id       = s.charger_id,
    t.member_id        = s.member_id,
    t.started_at       = s.started_at,
    t.ended_at         = s.ended_at,
    t.session_date     = s.session_date,
    t.status           = s.status,
    t.time_band        = s.time_band,
    t.energy_kwh       = s.energy_kwh,
    t.duration_minutes = s.duration_minutes,
    t.idle_minutes     = s.idle_minutes,
    t.amount_paid      = CAST(s.payment.amount_inr AS DECIMAL(10,2)),
    t.updated_at       = s.updated_at
WHEN NOT MATCHED THEN
  INSERT (
    session_id, charger_id, member_id, started_at, ended_at,
    session_date, status, time_band, energy_kwh, duration_minutes,
    idle_minutes, amount_paid, updated_at
  )
  VALUES (
    s.session_id, s.charger_id, s.member_id, s.started_at, s.ended_at,
    s.session_date, s.status, s.time_band, s.energy_kwh, s.duration_minutes,
    s.idle_minutes, CAST(s.payment.amount_inr AS DECIMAL(10,2)), s.updated_at
  )

""")